# deepVOL (CNN-LSTM) 全流程实现

本 Notebook 按照论文 *The short-term predictability of returns in order book markets: A deep learning perspective* 的 deepVOL 思路，从**按日 GZ/Parquet 的逐笔交易与 10 档 LOB 数据**构建：

1. 数据读取与清洗
2. deepVOL 体积空间特征构建
3. 平滑回报标签生成（动态阈值三分类）
4. 滑动窗口样本化 + Max-Scaling + 张量重塑
5. CNN + Inception + LSTM 的 deepVOL 模型
6. 训练与评估示例

> 说明：本 Notebook 默认以 `data/raw/orderbooks` 下的按日文件为输入；如果存在 gzip CSV（`.gz/.csv.gz`）或 parquet（`.parquet`）会自动识别。


In [ ]:
from __future__ import annotations

import math
import glob
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# ========== 配置区域 ==========
@dataclass
class Config:
    data_root: Path = Path('data')
    orderbook_dir: Path = Path('data/raw/orderbooks')

    levels: int = 10
    W: int = 10
    tick_size: float = 0.01

    h: int = 20
    k: int = 5

    T: int = 100
    train_stride: int = 10
    eval_stride: int = 1

    batch_size: int = 64
    lr: float = 1e-3
    epochs: int = 10
    train_ratio: float = 0.7
    val_ratio: float = 0.15
    seed: int = 42


cfg = Config()
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

print(cfg)


In [ ]:
def make_orderbook_columns(levels: int = 10) -> List[str]:
    cols = ['timestamp']
    for i in range(1, levels + 1):
        cols += [f'bidprice{i}', f'bidvolume{i}']
    for i in range(1, levels + 1):
        cols += [f'askprice{i}', f'askvolume{i}']
    return cols


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    expected = make_orderbook_columns(10)
    if len(df.columns) == len(expected) and ('timestamp' not in df.columns):
        df.columns = expected
    return df


In [ ]:
def load_one_orderbook_file(path: Path) -> pd.DataFrame:
    suffix = ''.join(path.suffixes).lower()
    if suffix.endswith('.parquet'):
        df = pd.read_parquet(path)
    elif suffix.endswith('.csv') or suffix.endswith('.csv.gz') or suffix.endswith('.gz'):
        try:
            df = pd.read_csv(path)
        except Exception:
            df = pd.read_csv(path, header=None)
    else:
        raise ValueError(f'Unsupported file type: {path}')
    return normalize_columns(df)


def list_orderbook_files(orderbook_dir: Path) -> List[Path]:
    files: List[Path] = []
    for p in ['*.parquet', '*.csv', '*.gz', '*.csv.gz']:
        files += [Path(x) for x in glob.glob(str(orderbook_dir / p))]
    return sorted(set(files))


files = list_orderbook_files(cfg.orderbook_dir)
print(f'发现 {len(files)} 个 orderbook 文件')
for x in files[:5]:
    print(' -', x)


In [ ]:
def clean_orderbook(df: pd.DataFrame) -> pd.DataFrame:
    required = ['timestamp', 'bidprice1', 'askprice1']
    for c in required:
        if c not in df.columns:
            raise KeyError(f'Missing required column: {c}')

    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    out = out.dropna(subset=['timestamp', 'bidprice1', 'askprice1'])
    out = out[out['askprice1'] > out['bidprice1']]
    out = out.sort_values('timestamp').groupby('timestamp', as_index=False).last()

    t0, t1 = out['timestamp'].min(), out['timestamp'].max()
    if t1 > t0:
        left = t0 + (t1 - t0) * (10 / (24 * 60))
        right = t1 - (t1 - t0) * (10 / (24 * 60))
        out = out[(out['timestamp'] >= left) & (out['timestamp'] <= right)]

    return out.reset_index(drop=True)


In [ ]:
def build_volume_features(clean_df: pd.DataFrame, W: int, tick_size: float, levels: int = 10) -> np.ndarray:
    N = len(clean_df)
    feat = np.zeros((N, 2 * W), dtype=np.float32)

    bps = [f'bidprice{i}' for i in range(1, levels + 1)]
    bvs = [f'bidvolume{i}' for i in range(1, levels + 1)]
    aps = [f'askprice{i}' for i in range(1, levels + 1)]
    avs = [f'askvolume{i}' for i in range(1, levels + 1)]

    for t in range(N):
        row = clean_df.iloc[t]
        mid = (row['askprice1'] + row['bidprice1']) / 2.0

        for p_col, v_col in zip(bps, bvs):
            p, v = row[p_col], row[v_col]
            if pd.isna(p) or pd.isna(v):
                continue
            distance = int(math.floor((mid - p) / tick_size))
            if 1 <= distance <= W:
                feat[t, W - distance] += float(v)

        for p_col, v_col in zip(aps, avs):
            p, v = row[p_col], row[v_col]
            if pd.isna(p) or pd.isna(v):
                continue
            distance = int(math.floor((p - mid) / tick_size))
            if 1 <= distance <= W:
                feat[t, W + distance - 1] += float(v)

    return feat


In [ ]:
def smooth_future_mid(mid: pd.Series, h: int, k: int) -> pd.Series:
    return mid.rolling(window=2*k+1, center=True, min_periods=2*k+1).mean().shift(-h)


def compute_returns(mid: pd.Series, h: int, k: int) -> pd.Series:
    future = smooth_future_mid(mid, h, k)
    return (future - mid) / mid


def compute_gamma(train_returns: np.ndarray) -> float:
    train_returns = train_returns[~np.isnan(train_returns)]
    q33, q66 = np.quantile(train_returns, [0.33, 0.66])
    return float((abs(q33) + q66) / 2.0)


def discretize_labels(returns: np.ndarray, gamma: float) -> np.ndarray:
    y = np.full_like(returns, fill_value=-1, dtype=np.int64)
    valid = ~np.isnan(returns)
    y[(returns < -gamma) & valid] = 0
    y[(np.abs(returns) <= gamma) & valid] = 1
    y[(returns > gamma) & valid] = 2
    return y


In [ ]:
def create_dataset(volume_features: np.ndarray, labels: np.ndarray, T: int, W: int, stride: int = 1) -> Tuple[np.ndarray, np.ndarray]:
    X_list, y_list = [], []

    for t in range(T, len(volume_features), stride):
        y_t = labels[t]
        if y_t < 0:
            continue

        window = volume_features[t-T:t]
        local_max = np.max(window)
        if local_max <= 0:
            continue

        window = window / local_max
        bid = window[:, :W]
        ask = window[:, W:]
        x = np.stack([bid, ask], axis=-1).astype(np.float32)

        X_list.append(x)
        y_list.append(y_t)

    if len(X_list) == 0:
        return np.empty((0, T, W, 2), dtype=np.float32), np.empty((0,), dtype=np.int64)

    return np.stack(X_list, axis=0), np.asarray(y_list, dtype=np.int64)


In [ ]:
def prepare_one_day(path: Path, cfg: Config):
    raw = load_one_orderbook_file(path)
    clean = clean_orderbook(raw)

    if len(clean) < cfg.T + cfg.h + 2*cfg.k + 5:
        return None

    feat = build_volume_features(clean, W=cfg.W, tick_size=cfg.tick_size, levels=cfg.levels)
    mid = (clean['askprice1'] + clean['bidprice1']) / 2.0
    rets = compute_returns(mid, h=cfg.h, k=cfg.k).values

    N = len(clean)
    tr_end = int(N * cfg.train_ratio)
    va_end = int(N * (cfg.train_ratio + cfg.val_ratio))

    gamma = compute_gamma(rets[:tr_end])
    labels = discretize_labels(rets, gamma)

    X_train, y_train = create_dataset(feat[:tr_end], labels[:tr_end], cfg.T, cfg.W, cfg.train_stride)
    X_val, y_val = create_dataset(feat[tr_end:va_end], labels[tr_end:va_end], cfg.T, cfg.W, cfg.eval_stride)
    X_test, y_test = create_dataset(feat[va_end:], labels[va_end:], cfg.T, cfg.W, cfg.eval_stride)

    info = {
        'path': str(path),
        'rows_raw': len(raw),
        'rows_clean': len(clean),
        'gamma': gamma,
        'train_samples': len(y_train),
        'val_samples': len(y_val),
        'test_samples': len(y_test),
    }
    return (X_train, y_train, X_val, y_val, X_test, y_test, info)


In [ ]:
all_parts = []
infos = []

for f in files:
    out = prepare_one_day(f, cfg)
    if out is None:
        continue
    Xtr, ytr, Xva, yva, Xte, yte, info = out
    if len(ytr) == 0:
        continue
    all_parts.append((Xtr, ytr, Xva, yva, Xte, yte))
    infos.append(info)

if len(all_parts) == 0:
    print('未生成样本。请检查数据路径、tick_size、最小样本长度参数。')
else:
    X_train = np.concatenate([p[0] for p in all_parts], axis=0)
    y_train = np.concatenate([p[1] for p in all_parts], axis=0)
    X_val = np.concatenate([p[2] for p in all_parts], axis=0)
    y_val = np.concatenate([p[3] for p in all_parts], axis=0)
    X_test = np.concatenate([p[4] for p in all_parts], axis=0)
    y_test = np.concatenate([p[5] for p in all_parts], axis=0)

    print('样本汇总:')
    print('X_train:', X_train.shape, 'y_train:', y_train.shape)
    print('X_val  :', X_val.shape, 'y_val  :', y_val.shape)
    print('X_test :', X_test.shape, 'y_test :', y_test.shape)

    display(pd.DataFrame(infos).head())


In [ ]:
class DeepVOLDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float().unsqueeze(1)
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
class InceptionTime(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        b = out_ch // 4
        self.b1 = nn.Conv2d(in_ch, b, kernel_size=(1, 1), padding=(0, 0))
        self.b3 = nn.Conv2d(in_ch, b, kernel_size=(3, 1), padding=(1, 0))
        self.b5 = nn.Conv2d(in_ch, b, kernel_size=(5, 1), padding=(2, 0))
        self.bp = nn.Sequential(
            nn.MaxPool2d(kernel_size=(3, 1), stride=1, padding=(1, 0)),
            nn.Conv2d(in_ch, out_ch - 3*b, kernel_size=(1, 1), padding=(0, 0)),
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)
        return self.act(x)


class DeepVOLNet(nn.Module):
    def __init__(self, W: int = 10, n_classes: int = 3, lstm_hidden: int = 64):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=(1, 1, 2), stride=(1, 1, 1)),
            nn.ReLU(inplace=True),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1, 2), padding=(0, 0)),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=(3, 1), padding=(1, 0)),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=(1, 2), padding=(0, 0)),
            nn.ReLU(inplace=True),
        )
        self.inception = InceptionTime(32, 64)
        self.lstm = nn.LSTM(input_size=64, hidden_size=lstm_hidden, batch_first=True)
        self.fc = nn.Linear(lstm_hidden, n_classes)

    def forward(self, x):
        x = self.conv3d(x)
        x = x.squeeze(-1)
        x = self.spatial(x)
        x = self.inception(x)
        x = x.mean(dim=-1)
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device='cpu'):
    train = optimizer is not None
    model.train() if train else model.eval()

    loss_sum, n, correct = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            logits = model(X)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        bs = y.size(0)
        loss_sum += loss.item() * bs
        n += bs
        correct += (logits.argmax(1) == y).sum().item()

    return loss_sum / max(n, 1), correct / max(n, 1)


def predict(model, loader, device='cpu'):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for X, y in loader:
            logits = model(X.to(device))
            ys.append(y.numpy())
            ps.append(logits.argmax(1).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

if 'X_train' in globals() and len(y_train) > 0:
    train_loader = DataLoader(DeepVOLDataset(X_train, y_train), batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(DeepVOLDataset(X_val, y_val), batch_size=cfg.batch_size, shuffle=False)
    test_loader = DataLoader(DeepVOLDataset(X_test, y_test), batch_size=cfg.batch_size, shuffle=False)

    model = DeepVOLNet(W=cfg.W).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    best_val, best_state = float('inf'), None
    for ep in range(1, cfg.epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = run_epoch(model, val_loader, criterion, None, device)
        print(f'Epoch {ep:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}')
        if va_loss < best_val:
            best_val = va_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    y_true, y_pred = predict(model, test_loader, device)
    print('
[TEST] Confusion Matrix')
    print(confusion_matrix(y_true, y_pred))
    print('
[TEST] Classification Report')
    print(classification_report(y_true, y_pred, digits=4))

    Path('models').mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), 'models/deepvol_cnn_lstm.pt')
    print('模型已保存到 models/deepvol_cnn_lstm.pt')
else:
    print('当前未构建出训练样本，跳过训练。')


## 可选扩展建议

- 按 `exchange-symbol-date` 分层切分，避免同一交易日泄露。
- 在训练集网格搜索 `tick_size / W / T / h`。
- 多任务头：同时预测方向与波动强度。
- 引入 `trades` 分支（逐笔成交量、方向）与 LOB 分支做融合。
